# Práctica de RAG naive — del LLM puro al pipeline completo

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/practica_completa.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** construir un sistema RAG naive de punta a punta — corpus, chunking, embeddings, ChromaDB, retrieval, augmentation, generation — sobre el programa real de la Cátedra IA UTN-FRSF 2026, y validar que efectivamente trae información que el LLM puro no podría saber.
>
> **Por qué este corpus:** el programa de la cátedra (docentes, ordenanzas, fechas, umbrales de aprobación) contiene datos **imposibles de adivinar** para el LLM. Si el RAG funciona, los va a recuperar; si el LLM puro intenta responderlos, los inventa.
>
> **Estructura:**
> - **Parte 1** — Setup + motivación (LLM puro vs RAG hardcoded).
> - **Parte 2** — Pipeline RAG naive completo a mano (corpus → chunking → ChromaDB → retrieval → augmentation → generation).
> - **Parte 3** — Probamos el pipeline con tres queries de distinto tipo.
>
> **¿Y hybrid + reranking?** Esos los ves en `practica_legal.ipynb` aplicados sobre un corpus jurídico real (Ley 21.526 de Entidades Financieras), donde tiene sentido subir el nivel del retrieval.
>
> **Tiempo:** ~30 min de hands-on.

# Parte 1 — Setup + motivación

Antes de meter retrieval automático, veamos qué pasa cuando le preguntamos al LLM cosas que **no puede saber**.


## 0. Setup

Instalá dependencias y configurá la API key de Groq.


In [ ]:
!pip install -q sentence-transformers chromadb rank_bm25 groq numpy pandas matplotlib


In [ ]:
# OPCIONAL — sólo para correr en local. En Colab esta celda se autodetecta y se omite
# (en Colab configurá GROQ_API_KEY desde "Secrets" en la barra lateral izquierda).
#
# Si vos corrés esta notebook localmente con tu venv:
#   1. Poné GROQ_API_KEY=tu-api-key en un archivo .env en la raíz del repo
#   2. Esta celda lo carga automáticamente

try:
    import google.colab  # noqa: F401
    print('ℹ️  Colab detectado — saltando .env (usá Colab Secrets para GROQ_API_KEY).')
except ImportError:
    try:
        from dotenv import load_dotenv, find_dotenv
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
        from dotenv import load_dotenv, find_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f'✓ .env cargado desde {env_path}')
    else:
        print('ℹ️  No se encontró .env. Asegurate de exportar GROQ_API_KEY como env var antes de lanzar jupyter.')


In [ ]:
import os

try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')


In [ ]:
from groq import Groq

_groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])


def llamar_llm(messages, model='llama-3.1-8b-instant', temperature=0.3, max_tokens=1024):
    """Wrapper unificado para llamar Groq — el mismo de Clase 2."""
    resp = _groq_client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


print(llamar_llm([{'role': 'user', 'content': 'Decime "ok" si me podés escuchar'}]))


In [ ]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f'✓ Modelo de embeddings: {modelo_emb.get_sentence_embedding_dimension()} dims')


## 1. Motivación — el LLM no puede saber esto

**Pregunta concreta:** *"¿Cuándo es el parcial de Inteligencia Artificial 2026 de la UTN-FRSF y qué tema cubre?"*

Esto **no está en el training del LLM**. Vamos a ver:
1. Qué responde el LLM puro — muy probablemente **inventa con confianza**.
2. Qué responde con contexto manual (RAG hardcoded simulado).

Después automatizamos la búsqueda del contexto.


In [ ]:
pregunta = '¿Cuándo es el parcial de Inteligencia Artificial 2026 de la UTN-FRSF y qué tema cubre?'

# Hechos verificables (sacamos "06/04" porque el LLM va a escribir "6 de abril" en prosa)
hechos_verdaderos = ['6 de abril', 'Búsqueda en espacio de estado', 'campus virtual']


def score_keywords(respuesta, keywords):
    return sum(1 for k in keywords if k.lower() in respuesta.lower())


# ── 1) LLM PURO (sin contexto) ──
resp_sin = llamar_llm([
    {'role': 'system', 'content':
        'Respondé en español, máximo 4 oraciones. Sé directo: si tenés información, dala; '
        'si no tenés, decilo breve.'},
    {'role': 'user', 'content': pregunta},
], temperature=0.3)
score_sin = score_keywords(resp_sin, hechos_verdaderos)

print('╔════════════════════════════════════════════════════════════╗')
print('║   LLM PURO (sin RAG)                                       ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_sin)
print()
print(f'   Score de hechos correctos: {score_sin}/{len(hechos_verdaderos)}')
print('   ⚠️  Dos resultados posibles:')
print('       (a) admite no saber → score 0 pero el usuario queda sin respuesta')
print('       (b) inventa fechas o temas plausibles → score 0 y el usuario es engañado')
print('   En ambos casos: el LLM puro no resuelve el problema del usuario.')
print('╚════════════════════════════════════════════════════════════╝')

# ── 2) CON contexto manual explícito ──
# Clave: el contexto debe ser INEQUÍVOCO sobre a qué materia/año se refiere.
contexto_manual = (
    'Contexto sobre la cátedra Inteligencia Artificial — UTN-FRSF — 1er cuatrimestre 2026: '
    'El parcial de la materia tiene fecha 6 de abril de 2026 y cubre el tema '
    '"Búsqueda en espacio de estado". Se rinde en clase usando el campus virtual.'
)
resp_con = llamar_llm([
    {'role': 'system', 'content':
        'Sos un asistente de la cátedra. Respondé la pregunta usando la información del '
        'contexto provisto. Sé concreto, citá los datos exactos. Máximo 4 oraciones.'},
    {'role': 'user', 'content': f'Contexto:\n{contexto_manual}\n\nPregunta: {pregunta}'},
], temperature=0.3)
score_con = score_keywords(resp_con, hechos_verdaderos)

print()
print('╔════════════════════════════════════════════════════════════╗')
print('║   CON contexto (RAG hardcoded)                             ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_con)
print()
print(f'   ✓ Score de hechos correctos: {score_con}/{len(hechos_verdaderos)}')
print('   ✓ Información trazable al contexto provisto.')
print('╚════════════════════════════════════════════════════════════╝')

print()
print(f'💡 Diferencia visible: {score_con} vs {score_sin}. El RAG no es "más inteligente" —')
print('   tiene acceso a información que el LLM literalmente no puede saber.')
print('   Ahora vamos a automatizar la búsqueda del contexto.')


## 1b. ¿Y si el LLM **igual responde** sin saber? — la alucinación silenciosa

El caso del parcial el LLM lo manejó bien (admitió no saber). Pero hay preguntas donde el LLM **no se contiene** y responde **con confianza pero mal**.

Probemos con una pregunta procedural sobre las prácticas:


In [ ]:
pregunta_b = '¿Cuántos integrantes pueden tener los grupos de trabajos prácticos en la cátedra Inteligencia Artificial de UTN-FRSF? ¿Y cómo se inscriben los grupos?'

# Ground truth de nuestro corpus: 2 o 3 integrantes (1 < n ≤ 3), inscripción por email
hechos_b = ['2', '3', 'email']

resp_b_sin = llamar_llm([
    {'role': 'system', 'content':
        'Respondé en español, máximo 4 oraciones. Sé directo y concreto. Si tenés información, dala.'},
    {'role': 'user', 'content': pregunta_b},
], temperature=0.3)
score_b_sin = score_keywords(resp_b_sin, hechos_b)

print('╔════════════════════════════════════════════════════════════╗')
print('║   LLM PURO — pregunta procedural sin info real             ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_b_sin)
print()
print(f'   Score: {score_b_sin}/{len(hechos_b)}')
print('   ⚠️  Fijate si el LLM:')
print('       - Inventó un rango de integrantes (común: 3-5 o "depende de la cátedra")')
print('       - Inventó un método de inscripción (campus, sistema, formulario)')
print('   Ese es el problema MÁS PELIGROSO: una respuesta plausible pero falsa.')
print('╚════════════════════════════════════════════════════════════╝')

# Con contexto manual
contexto_b = (
    'Contexto sobre la cátedra Inteligencia Artificial — UTN-FRSF — 2026: '
    'Los trabajos prácticos y las actividades en clase se realizan en forma grupal. '
    'Los grupos están formados por 2 o 3 integrantes (1 < nro-integrantes ≤ 3). '
    'Cada grupo debe enviar por email su formación indicando nombre, apellido y '
    'dirección de email de todos los integrantes.'
)
resp_b_con = llamar_llm([
    {'role': 'system', 'content':
        'Sos un asistente de la cátedra. Respondé usando el contexto. Sé concreto, citá los datos exactos. Máximo 4 oraciones.'},
    {'role': 'user', 'content': f'Contexto:\n{contexto_b}\n\nPregunta: {pregunta_b}'},
], temperature=0.3)
score_b_con = score_keywords(resp_b_con, hechos_b)

print()
print('╔════════════════════════════════════════════════════════════╗')
print('║   CON contexto                                              ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_b_con)
print()
print(f'   ✓ Score: {score_b_con}/{len(hechos_b)}')
print('╚════════════════════════════════════════════════════════════╝')

print()
print(f'💡 La pregunta es: ¿el LLM puro **inventó** algo en su respuesta?')
print('   Si la respuesta contiene un número distinto a 2 o 3, o menciona inscripción')
print('   por sistema/campus en vez de email → ESO ES ALUCINACIÓN.')
print('   El RAG con contexto responde con los datos exactos del corpus.')


## 1c. Reflexión — la alucinación moderna es **más sutil** (y más peligrosa)

Si corriste la celda anterior con un modelo moderno (Llama 3.1/3.3, GPT-4+, Claude), es posible que hayas visto un patrón parecido a este:

> *"No tengo información actualizada sobre la estructura específica... **Sin embargo**, en general los grupos de trabajo pueden tener entre 3 a 6 integrantes... Para inscribirse, consultar el sitio web... presentar solicitud en secretaría..."*

### ¿Qué podría haber pasado?

| Comportamiento posible | Análisis |
|---|---|
| El modelo empieza con un disclaimer ("no tengo info específica") | ✓ Calibración correcta — sabe que no sabe |
| **Pero igual responde** con info "general" | ✗ Eso **sería** alucinación, con disfraz |
| Da números específicos (ej. 3-6) | ✗ La verdad es 2-3 |
| Da método de inscripción específico (web / secretaría) | ✗ La verdad es email |
| Sugiere verificar con la universidad | ✓ Buen consejo pero llega tarde |

### La alucinación "descarada" de 2023 ya casi no existe

Hasta 2023 los LLMs alucinaban **sin pudor** ("el parcial es el 15 de abril y cubre redes neuronales convolucionales" — todo inventado, sin disclaimer). Era fácil de detectar visualmente.

Con RLHF (Clase 2) los modelos aprendieron a **advertir su incertidumbre primero, inventar igual después**. Eso es **más peligroso** porque:

1. El disclaimer da una sensación falsa de calibración ("el modelo me avisa cuando no sabe").
2. La info inventada cuela igual, escondida tras el "en general".
3. El usuario no puede distinguir "info general verdadera" de "info general inventada".

### Cuándo el score podría mentir

Supongamos que el LLM respondió algo como *"entre 3 y 6 integrantes"*. El `score_keywords` matchearía el `'3'` aunque la respuesta **sería incorrecta** (la verdad es 2 o 3, no 3 a 6). En ese caso el score `2/3` estaría diciendo "67% correcto" cuando en realidad fue 0% correcto.

Este sería exactamente el problema que motiva **LLM-as-judge** (Clase 3b): el match por substring puede engañar. Un juez LLM podría leer la respuesta, distinguir "2 o 3" de "3 a 6", y darle el score real (0).

### El takeaway

Demostrar la "alucinación dramática" en clase con modelos modernos requiere preguntas más sutiles — o aceptar que el contraste viene de **info general inventada** + **disclaimer aparentemente honesto**. En cualquier caso, **el RAG resuelve el problema de raíz**: el modelo deja de adivinar porque tiene la respuesta exacta en el contexto.


# Parte 2 — Pipeline RAG naive completo

Reemplazamos el "contexto manual" del paso anterior por un sistema automático:
**corpus → chunking → embeddings → vector store → retrieval → augmentation → generation**.


## 2. El corpus — programa real de la Cátedra IA 2026

5 documentos sobre el programa de la materia. Los datos son específicos: docentes con cargo, fechas, ordenanzas, umbrales, aula. **El LLM puro no puede inventar nada de esto correctamente.**


In [ ]:
# ── Corpus: Cátedra Inteligencia Artificial — UTN-FRSF, 1er cuatrimestre 2026 ──

DOCUMENTOS = [
    {
        "id": "doc_identidad",
        "titulo": "Identidad y modalidad de la cátedra",
        "contenido": (
            "La asignatura Inteligencia Artificial pertenece al 5to año de la carrera "
            "Ingeniería en Sistemas de Información (ISI) de la Universidad Tecnológica "
            "Nacional, Facultad Regional Santa Fe (UTN-FRSF), y se dicta durante el "
            "1er cuatrimestre del ciclo 2026. La cátedra está radicada en el CIDISI "
            "(Centro de Investigación de Desarrollo e Innovación en Sistemas de "
            "Información). Los docentes a cargo son la Profesora Titular Milagros "
            "Gutiérrez y el Profesor Adjunto Jorge Roa. La cátedra no cuenta con "
            "auxiliares ni ayudantes alumnos en esta cohorte. La materia se dicta "
            "de manera presencial los días lunes de 18:00 a 21:00 horas en el "
            "Laboratorio 1 de la facultad. La modalidad de cursado es por proyecto, "
            "con foco en aplicación práctica más que en exposición magistral. La "
            "cátedra opera bajo el marco normativo de la Ordenanza 1877, que "
            "establece las competencias específicas (CE) y genéricas (CG) requeridas "
            "para todas las ingenierías de la UTN. Las competencias específicas "
            "incluyen CE1.1 (especificar y desarrollar sistemas de información), "
            "CE1.3 (desarrollar software para soluciones informáticas), CE4.1 "
            "(certificar funcionamiento de sistemas) y CE5.1 (dirigir implementación "
            "de sistemas). Las competencias genéricas incluyen CG4 (uso de técnicas "
            "y herramientas), CG5 (generar desarrollos tecnológicos), CG8 (ética y "
            "responsabilidad profesional) y CG9 (aprendizaje continuo)."
        ),
    },
    {
        "id": "doc_evaluacion",
        "titulo": "Regularidad, promoción directa y asistencia",
        "contenido": (
            "Para regularizar el cursado de Inteligencia Artificial el estudiante "
            "debe: (a) aprobar los dos trabajos prácticos con un puntaje igual o "
            "superior a 4, (b) aprobar el parcial con un puntaje igual o superior "
            "a 4, y (c) realizar las actividades de evaluación continua y participar "
            "en el debate sobre ética en IA. Adicionalmente se requiere el 80 por "
            "ciento de asistencia. Las condiciones para la promoción directa son "
            "las mismas, pero los puntajes mínimos suben a 6 tanto en los dos "
            "trabajos prácticos como en el parcial; también se requiere haber "
            "realizado las actividades de evaluación continua y haber participado "
            "del debate, manteniendo el 80 por ciento de asistencia. La nota final "
            "es el promedio de las notas de los trabajos prácticos, las evaluaciones "
            "continuas y el parcial. Los alumnos que demuestren niveles mínimos y "
            "básicos de aprendizaje pero no hayan alcanzado la aprobación directa "
            "rendirán un examen final escrito que abarca todos los temas del "
            "programa."
        ),
    },
    {
        "id": "doc_tps",
        "titulo": "Trabajos prácticos y modalidad de agrupamiento",
        "contenido": (
            "La materia tiene dos trabajos prácticos. El TP1 consiste en diseñar un "
            "clasificador neuronal y se realiza en conjunto con la cátedra de "
            "Ciencia de Datos. El TP2 consiste en diseñar un agente con temática "
            "libre y se divide en tres etapas: la primera etapa es la definición "
            "del agente y los objetivos de diseño; la segunda etapa es la "
            "definición de percepciones y acciones del agente más la descripción "
            "del ambiente donde actúa, junto con implementaciones parciales; la "
            "tercera etapa es la entrega final con código, presentación e informe. "
            "Los trabajos prácticos y las actividades en clase se realizan en forma "
            "grupal. Los grupos están formados por 2 o 3 integrantes — es decir, "
            "1 menor estricto que nro-integrantes menor o igual que 3. Cada grupo "
            "debe enviar por email su formación indicando nombre, apellido y "
            "dirección de email de todos los integrantes. Los aspectos a evaluar "
            "en cada TP son: el informe técnico y la presentación, la definición y "
            "alcance de los objetivos del TP, la presentación de la solución de "
            "software, la cantidad y calidad de las herramientas que se apliquen, "
            "la calidad y cantidad de pruebas de validación, y la expresividad en "
            "los coloquios de presentación de los prácticos."
        ),
    },
    {
        "id": "doc_cronograma",
        "titulo": "Programación de actividades 1er cuatrimestre 2026",
        "contenido": (
            "La programación de actividades para el 1er cuatrimestre 2026 es la "
            "siguiente. El debate en equipos sobre ética en IA se realiza el 16 de "
            "marzo. El parcial tiene fecha 6 de abril y cubre el tema Búsqueda en "
            "espacio de estado, y se rinde en clase usando el campus virtual. El "
            "TP1 sobre el desarrollo del modelo neuronal tiene fecha de entrega "
            "el 18 de mayo. El TP2 se desarrolla en tres etapas con entregas "
            "sucesivas: la primera etapa, definición del agente y objetivos de "
            "diseño, vence el 8 de junio; la segunda etapa, definición de "
            "percepciones y acciones del agente con descripción del ambiente e "
            "implementaciones parciales, vence el 22 de junio; la entrega final "
            "con código, presentación e informe vence el 29 de junio. Las clases "
            "del cuatrimestre se dictan los lunes de 18 a 21 horas en el "
            "Laboratorio 1."
        ),
    },
    {
        "id": "doc_recursos",
        "titulo": "Bibliografía, software y canal de comunicación",
        "contenido": (
            "La bibliografía principal de la materia incluye: Inteligencia "
            "Artificial Un enfoque moderno de Russell y Norvig, disponible en "
            "aima.cs.berkeley.edu; Inteligencia Artificial Segunda Edición de Rich "
            "y Knight; Inteligencia Artificial Una nueva síntesis de Nilsson; "
            "MultiAgent Systems de Wooldridge; y los apuntes de clase elaborados "
            "por la cátedra. El software utilizado incluye Python como lenguaje "
            "principal, notebooks Jupyter para los desarrollos y Google Colab "
            "como entorno de ejecución en la nube. El canal de comunicación oficial "
            "entre estudiantes y docentes es el campus virtual de la FRSF, "
            "disponible en https://campusvirtual.frsf.utn.edu.ar/, donde se "
            "publican los materiales, se administran los cuestionarios de "
            "evaluación continua y se entregan los trabajos prácticos. Los "
            "objetivos pedagógicos generales de la cátedra son: gestionar proyectos "
            "de construcción de sistemas inteligentes, reconocer estrategias de "
            "creación de sistemas inteligentes, resolver problemas de "
            "representación del conocimiento y razonamiento en ambientes "
            "deterministas y bajo incertidumbre, y evaluar e implementar modelos "
            "de aprendizaje automático para resolver problemas."
        ),
    },
]

print(f'✓ {len(DOCUMENTOS)} documentos de la cátedra IA cargados')
for doc in DOCUMENTOS:
    print(f'  - {doc["titulo"]} ({len(doc["contenido"])} chars)')


## 3. Chunking

Partimos cada documento en **oraciones** (split por `. ! ?`). Es lo más simple y funciona bien para textos de reglamento donde cada oración es autocontenida.


In [ ]:
import re


def chunk_por_oracion(texto):
    """Chunking por oración (split en punto/exclamación/interrogación + espacio)."""
    oraciones = re.split(r'(?<=[.!?])\s+', texto)
    return [o.strip() for o in oraciones if o.strip()]


all_chunks = []
all_ids = []
all_metadatas = []
for doc in DOCUMENTOS:
    chunks = chunk_por_oracion(doc['contenido'])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f'{doc["id"]}_chunk_{i}')
        all_metadatas.append({
            'titulo': doc['titulo'],
            'doc_id': doc['id'],
            'chunk_index': i,
        })

print(f'✓ {len(all_chunks)} chunks (oraciones) listas para indexar')
print(f'\nEjemplos:')
for chunk in all_chunks[:3]:
    print(f'  - "{chunk[:80]}..." ({len(chunk)} chars)')


## 4. Embeddings + ChromaDB (indexing)

Generamos un embedding por chunk y los almacenamos en ChromaDB in-memory.


In [ ]:
import chromadb

client = chromadb.Client()
try:
    client.delete_collection('clase3_docs')
except Exception:
    pass

collection = client.create_collection(
    name='clase3_docs',
    metadata={'hnsw:space': 'cosine'},
)

all_embeddings = modelo_emb.encode(all_chunks).tolist()

collection.add(
    documents=all_chunks,
    embeddings=all_embeddings,
    metadatas=all_metadatas,
    ids=all_ids,
)
print(f'✓ Indexados {collection.count()} chunks en ChromaDB')


## 5. Retrieval

Dada una query, buscamos los top-k chunks más similares por coseno.


In [ ]:
def buscar_chunks(query, n_results=3):
    """Retrieval denso: top-k por similitud coseno."""
    query_embedding = modelo_emb.encode([query]).tolist()
    return collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances'],
    )


query = '¿Quiénes son los docentes de la cátedra?'
results = buscar_chunks(query, n_results=3)
print(f'Query: "{query}"\n\nTop 3 chunks:')
for i in range(len(results['documents'][0])):
    titulo = results['metadatas'][0][i]['titulo']
    sim = 1 - results['distances'][0][i]
    print(f'  #{i+1} (sim {sim:.3f}) [{titulo}]: "{results["documents"][0][i][:80]}..."')


## 6. RAG end-to-end — `rag_query`

System prompt fuerza al LLM a **basarse SOLO en el contexto** y **citar fuentes**.


In [ ]:
SYSTEM_RAG = """Sos un asistente de la cátedra Inteligencia Artificial de UTN-FRSF
que responde preguntas basándose ÚNICAMENTE en el contexto proporcionado.

Reglas:
- Usá SOLO la información del contexto. Nunca inventes fechas, nombres o números.
- Si el contexto no tiene información suficiente, decí "No tengo información suficiente en los documentos proporcionados."
- Citá la fuente entre corchetes cuando sea posible, ej: [Cronograma 2026].
- Respondé en español, conciso (máximo 5 oraciones)."""


def rag_query(pregunta, n_chunks=3, verbose=False):
    """Pipeline RAG completo: retrieval → augmentation → generation."""
    results = buscar_chunks(pregunta, n_results=n_chunks)

    contexto_partes = []
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        titulo = results['metadatas'][0][i]['titulo']
        contexto_partes.append(f'[{i+1}] ({titulo}): {doc}')
    contexto = '\n\n'.join(contexto_partes)

    messages = [
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {pregunta}'},
    ]
    respuesta = llamar_llm(messages, temperature=0.3)

    if verbose:
        print(f'Pregunta: {pregunta}\n')
        print(f'📎 Chunks recuperados:')
        for parte in contexto_partes:
            print(f'  {parte[:100]}...')
        print(f'\n💬 Respuesta:\n{respuesta}')

    return respuesta, results


respuesta, _ = rag_query('¿Cuáles son las fechas de entrega del TP2?', verbose=True)


# Parte 3 — Probemos el pipeline con algunas queries

## 7. ¿Funciona el RAG naive?

Vamos a correr `rag_query()` sobre **tres preguntas** que ejercitan distintas situaciones:

1. **Lookup directo** — la respuesta está literalmente en un sólo chunk del corpus.
2. **Multi-fact** — la respuesta requiere combinar información de varios chunks (por ejemplo, criterios de aprobación que cruzan TPs + parcial + asistencia).
3. **Edge case** — la información **no está** en el corpus. Esperamos que el sistema se abstenga, no que invente.

In [ ]:
preguntas = [
    '¿Qué día, horario y aula se dicta la materia?',
    '¿Qué necesito hacer para aprobar la materia?',
    '¿Cuándo es la entrega del TP3?',
]

for i, pregunta in enumerate(preguntas, 1):
    print('═' * 80)
    print(f'  Query {i}: {pregunta}')
    print('═' * 80)
    respuesta, results = rag_query(pregunta, n_chunks=3, verbose=False)
    docs_recuperados = [m['doc_id'] for m in results['metadatas'][0]]
    print(f'Docs recuperados: {docs_recuperados}')
    print()
    print('Respuesta:')
    print(respuesta)
    print()


### Lo que acabás de ver

Sobre las tres queries:

1. **Lookup directo (modalidad cursada)** — el sistema debería citar el día, horario y aula con precisión (lunes 18-21 hs, Laboratorio 1). Es un caso donde naive RAG **alcanza** porque la query y el chunk comparten vocabulario natural.
2. **Multi-fact (aprobar la materia)** — el sistema debería integrar TPs ≥4, parcial ≥4, evaluación continua, participación en el debate, 80% de asistencia. Si trae info parcial o se queda con un sólo aspecto, ahí es donde haría falta subir el nivel del retrieval (hybrid, reranking).
3. **Edge case (TP3)** — la cátedra tiene **dos** TPs (TP1 + TP2), no tres. La respuesta correcta es admitir que no hay información sobre un "TP3". Si inventa una fecha, eso es **alucinación pura** — el problema más peligroso de un sistema RAG.

### El paso obligado

¿Cómo medir esto sistemáticamente cuando tenés cientos de queries y varios pipelines candidatos? Esa es la pregunta de **Clase 3b**: LLM-as-judge, RAGAS, observability con Arize AX, y red teaming.

¿Cómo subir el nivel cuando naive no alcanza? Eso lo ves en **`practica_legal.ipynb`** — el mismo pipeline aplicado a un corpus jurídico real (Ley 21.526 de Entidades Financieras), donde sumamos BM25 + RRF para los casos que dense solo no maneja.